# Treasuries data workflow

`pull_treasuries` returns CRSP daily treasuries for a date range. Each business day is cached to `data/raw/treasuries/crsp_treasuries_YYYYMMDD.parquet`, so WRDS is only hit for days not yet cached.

In [8]:
import logging
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, so `src` imports

from src.data import pull_treasuries

logging.basicConfig(level=logging.INFO, format="%(message)s")

## Pull

First call queries WRDS and caches per day. Later calls read the cache. `force_refresh=True` re-pulls.

In [9]:
df = pull_treasuries("2024-01-01", "2024-01-31")
df.head()

querying WRDS 2024-01-02 to 2024-01-31


WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


ArrowKeyError: No type extension with name arrow.py_extension_type found

## Sanity checks

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9248 entries, 0 to 9247
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   cusip               9248 non-null   string        
 1   date                9248 non-null   datetime64[ns]
 2   security_type       9248 non-null   int64         
 3   coupon_rate         9248 non-null   Float64       
 4   dated_date          9248 non-null   datetime64[ns]
 5   maturity_date       9248 non-null   datetime64[ns]
 6   bid_price           9248 non-null   Float64       
 7   ask_price           9248 non-null   Float64       
 8   mid_price           9248 non-null   Float64       
 9   mid_price_flag      9248 non-null   string        
 10  accrued_interest    9248 non-null   Float64       
 11  daily_yield         8154 non-null   Float64       
 12  annual_yield        8154 non-null   Float64       
 13  duration_days       8154 non-null   Float64     

In [ ]:
df.groupby("date").size().describe()

count     21.000000
mean     440.380952
std        1.359272
min      439.000000
25%      439.000000
50%      440.000000
75%      441.000000
max      443.000000
dtype: float64

In [ ]:
df.groupby("security_type").size()

security_type
1     1995
2     5076
4     1083
11     462
12     632
dtype: int64

In [ ]:
df[["bid_price", "ask_price", "mid_price", "annual_yield", "duration_years"]].describe()

,bid_price,ask_price,mid_price,annual_yield,duration_years
count,9248.0,9248.0,9248.0,8154.0,8154.0
mean,93.789519,93.830988,93.810254,0.044272,4.960763
std,9.757897,9.74448,9.75117,0.004781,5.464816
min,49.984375,50.078125,50.03125,0.001239,0.00274
25%,91.394531,91.429688,91.410156,0.040437,0.909847
50%,97.171875,97.195312,97.182151,0.042941,2.761797
75%,99.332031,99.359375,99.351562,0.047541,5.960571
max,113.101562,113.148438,113.125,0.077129,20.654885
